# 🧮 Geometries for swimming embryo chimeras
This notebook provides tools for translating "organism-based" parameters into explicit geometric parameters for the [ChimeraDesign](./ChimeraDesign.ipynb) notebook, for simulation in the [ChimeraSwim](./ChimeraSwim.ipynb) model.

## Organism-based *vs.* geometrical parameters
The motivation for defining two sets of parameters is that they define larval morphologies in ways that lend themselves to two different purposes:
1. Organism-based parameters are intuitive to use in posing biological hypotheses, because they are expressed in terms of organismal constraints such as the tissue volume, and measurable functional traits such as excess density.
2. Geometrical parameters are necessary for constructing elements of the biomechanical model, but are unintuitive for posing biological hypotheses because they confound variations in multiple organismal traits.

Specifying either of these parameter sets also specifies the other.
Because the translations between them are not straightforward, they are implemeted in code below.

### Geometrical parameters
Larval morphologies in the [ChimeraSwim](./ChimeraSwim.ipynb) are defined by the geometrical parameters shown in this diagram:

![alt text](ChimeraSpheroid_geometry2.png "Surface shape parameters")

The parameters defining the external shape of the larva are:
- $D_s$: the maximum diameter of the external surface, at its equator
- $L_{s_1}$: the length of the top semi-spheroid (distance from the upper tip to the equator) of the external surface
- $L_{s_2}$: the length of the bottom semi-spheroid (distance from the lower tip to the equator) of the external surface

The parameters defining the shape of the internal "inclusion" are:
- $D_i$: the maximum diameter of the inclusion, at its equator
- $L_{i_1}$: the length of the top semi-spheroid (distance from the upper tip to the equator) of the inclusion
- $L_{i_2}$: the length of the bottom semi-spheroid (distance from the lower tip to the equator) of the inclusion
- $h_i$: the height of the inclusion equator above the external shape equator

It's assumed that the ambient seawater density is $1030 \frac{kg}{m^3}$.

### Organism-based parameters


In [1]:
# Set up graphics environment
%matplotlib widget
from matplotlib import pyplot
pyplot.ioff()
# Import modules
from math import pi
# Import widget infrastructure
from ipywidgets import interact, interactive, fixed, interact_manual, Output
import ipywidgets as widgets

from mpl_interactions import ipyplot as iplt
from mpl_toolkits import mplot3d
from matplotlib.colors import LightSource
# Import modules
import numpy as np
import os
# set up path to submodules
import sys
#sys.path.append('../../../submodules/')
#sys.path.append('../../../submodules/pyVRS')
# Import widget infrastructure
from IPython import display as idisplay
import pickle
from ipyfilechooser import FileChooser
from copy import deepcopy

In [2]:
# Import chimera model components
import pyVRSmorph as mrph
import pyVRSflow as flw
from meshSpheroid import chimeraSpheroid
# Define function converting organism-based to geometrical parameters
global Ds,Ls0,Ls1,Ls2,Di,Li0,Li1,Li2,hi,Vs,Vi,beta,xsi
global rho_tissue,rho_incl
def get_geom_pars(Vt,alpha,eta,rho_tissue_,rho_incl_,rho_excess,rho_seawater=1030,sigma = 0.95):
    global Ds,Ls0,Ls1,Ls2,Di,Li0,Li1,Li2,hi,Vs,Vi,beta,xsi
    global rho_tissue,rho_incl
    rho_tissue = rho_tissue_
    rho_incl = rho_incl_
    drho_tissue = rho_tissue - rho_seawater
    drho_incl = rho_incl - rho_seawater
    # calculate geometry of tissue-only chimera
    Dt = (6*Vt/(pi*alpha))**(1/3)
    Lt0 = alpha*Dt
    Lt2 = eta * Lt0
    Lt1 = Lt0 - Lt2
    # Compute "inflated surface after inclusion to adjust excess density
    beta = (drho_tissue-drho_incl)/(rho_excess-drho_incl)
    Ds = beta**(1/3) * Dt
    Vs = beta*Vt
    Ds =  (6*Vs/(pi*alpha))**(1/3)
    Ls0 = alpha*Ds
    Ls2 = eta * Ls0
    Ls1 = Ls0 - Ls2
    
    Vi = Vs - Vt
    Di =  (6*Vi/(pi*alpha))**(1/3)
    Li0 = alpha*Di
    Li2 = eta * Li0
    Li1 = Li0 - Li2
    
    xsi = (1-eta)*(sigma - ((beta-1)/beta)**(1/3))
    hi = xsi * Li0

    # Print results
    print('External surface parameters:')
    print(f'Ds = {Ds:.2e}')
    print(f'Ls0 = {Ls0:.2e}')
    print(f'Ls1 = {Ls1:.2e}')
    print(f'Ls2 = {Ls2:.2e}')
    
    print('\nInclusion parameters:')
    print(f'Di = {Di:.2e}')
    print(f'Li0 = {Li0:.2e}')
    print(f'Li1 = {Li1:.2e}')
    print(f'Li2 = {Li2:.2e}')
    print(f'hi = {hi:.2e}')
    
    print('\nAdditional information:')
    print(f'Vs = {Vs:.2e}')
    print(f'Vt = {Vt:.2e}')
    print(f'Vi = {Vi:.2e}\n')
    print(f'beta = {beta:.2e}')
    print(f'xsi = {xsi:.2e}\n')
    print(f'tissue excess density = {drho_tissue}')
    print(f'inclusion excess density = {drho_incl}')


In [3]:
global ds,nlevel0s,nlevel1s,di,nlevel0i,nlevel1i
ds=6e-6
nlevel0s=16
nlevel1s=12
_ds=widgets.FloatText(value=6.e-6,description = r"$d_s$")
_nlevel0s=widgets.IntText(value=12,description = r"$n_{s_0}$")
_nlevel1s=widgets.IntText(value=12,description = r"$n_{s_1}$")
_di=widgets.FloatText(value=6.e-6,description = r"$d_i$")
_nlevel0i=widgets.IntText(value=12,description = r"$n_{i_0}$")
_nlevel1i=widgets.IntText(value=12,description = r"$n_{i_1}$")

ui0r = widgets.VBox([_ds,_di])
ui1r = widgets.VBox([_nlevel0s,_nlevel0i])
ui2r = widgets.VBox([_nlevel0s,_nlevel0i])
ui012r = widgets.HBox([ui0r,ui1r,ui2r])

def set_res_pars(ds_,nlevel0s_,nlevel1s_,di_,nlevel0i_,nlevel1i_):
    global ds,nlevel0s,nlevel1s,di,nlevel0i,nlevel1i
    ds = ds_
    nlevel0s = nlevel0s_
    nlevel1s = nlevel1s_
    di = di_
    nlevel0i = nlevel0i_
    nlevel1i = nlevel1i_
    print('\nResolution parameters:')
    print(f'ds = {ds:.2e}')
    print(f'nlevel0s = {nlevel0s}')
    print(f'nlevel1s = {nlevel1s}\n')
    print(f'di = {ds:.2e}')
    print(f'nlevel0i = {nlevel0i}')
    print(f'nlevel1i = {nlevel1i}')


outr = widgets.interactive_output(set_res_pars,{'ds_':_ds,
                                                'nlevel0s_':_nlevel0s,
                                                'nlevel1s_':_nlevel1s,
                                                'di_':_di,
                                                'nlevel0i_':_nlevel0i,
                                                'nlevel1i_':_nlevel1i,})
display(ui012r,outr)

Output()

In [4]:
# Create textboxes for parameter input, setting default starting organismal parameters
_Vt=widgets.FloatText(value=(5.e-5)**3,width=10,description = r"$V_t$ ($m^3$)")
_alpha=widgets.FloatText(value=2.,description = r"$\alpha$")
_eta=widgets.FloatText(value=0.75,description = r"$\eta$")
_rho_tissue=widgets.FloatText(value=1070,description = r"$\rho_{tissue}$ ($\frac{kg}{m^3}$)")
_rho_incl=widgets.FloatText(value=1030,description = r"$\rho_{incl}$ ($\frac{kg}{m^3}$)")
_rho_excess=widgets.FloatText(value=25,description = r"$\rho_{excess}$ ($\frac{kg}{m^3}$)")

ui0s = widgets.VBox([_Vt,_alpha,_eta])
ui1s = widgets.VBox([_rho_tissue,_rho_incl,_rho_excess])
ui01s = widgets.HBox([ui0s,ui1s])


outs = widgets.interactive_output(get_geom_pars,{'Vt':_Vt,'alpha':_alpha,'eta':_eta,
                                                'rho_tissue_':_rho_tissue,
                                                'rho_incl_':_rho_incl,
                                                'rho_excess':_rho_excess})
display(ui01s,outs)

Output()

:::{figure} #cd_1
:placeholder: ./images/CG_1.png
:align: left
:::

### Visualize the new larval shape
Click the button to visualize the shape of the larva specified by the parameters you entered in the input boxes above. 
The output also includes textual information with technical details about the model larva.

If you want to make changes, go back up to the previous step to change parameters, then press the button again again to display the new shape.
When you're happy with the geometry, proceed to the next step.

In [5]:
button7 = widgets.Button(description="Visualize the shape")
output7 = widgets.Output()

buttonsV = widgets.VBox([button7, output7])
display(buttonsV)

@output7.capture()
def on_button_clicked7(b):
    global Mnew, surf_pars, incl1_pars
    global Ds,Ls0,Ls1,Ls2,Di,Li0,Li1,Li2,hi,Vs,Vi,beta,xsi
    global ds,nlevel0s,nlevel1s,di,nlevel0i,nlevel1i
    global rho_tissue,rho_incl
    # Define surface and inclusion surfaces
    CSsurf = chimeraSpheroid(D=Ds,L1=Ls1,L2=Ls2,d=ds,nlevels=[nlevel0s,nlevel1s])
    CSincl = chimeraSpheroid(D=Di,L1=Li1,L2=Li2,d=di,nlevels=[nlevel0i,nlevel1i],
                             translate=[0.,0.,hi])
    Mnew = mrph.Morphology()
    Mnew.check_normals = False
    Mnew.gen_surface(vectors=CSsurf.vectors)
    # materials parameter can be 'seawater', 'tissue', 'lipid' or 'calcite' 
    Mnew.gen_inclusion(vectors=CSincl.vectors,material='seawater',immersed_in=1)
    #Mnew.gen_inclusion(vectors=CEincl.vectors,material='freshwater',immersed_in=1)
    # Update the tissue and inclusion densities
    Mnew.layers[1].pars['density'] = rho_tissue
    Mnew.layers[2].pars['density'] = rho_incl
    
    figureM = pyplot.figure(num=57)
    axesM = figureM.add_subplot(projection='3d')
    Mnew.plot_layers(axes=axesM)
    
    figureM.canvas.draw()
    figureM.canvas.flush_events()
    pyplot.pause(0.25)
    
button7.on_click(on_button_clicked7)

### Compute the fluid forces
Executing this cell will compute the geometry and fluid flow around the model larva. 

Depending on the number of triangle and the speed of your computer, it may take a few seconds to complete: 
- When it starts calculating, it prints out "Calculating inverse...". 
- When the calculation is complete, it prints out "Done calculating inverse."
- The output includes statistics like the magnitudes and centers of buoyancy and gravity.

**You need to run this just once for each larval shape. You do not need to rerun it unless you change the larval parameters.**

In [6]:
button6 = widgets.Button(description="Calculate flow")
output6 = widgets.Output()

buttonsC = widgets.HBox([button6, output6])
display(buttonsC)

@output6.capture()
def on_button_clicked6(b):
    Mnew.body_calcs()
    Mnew.flow_calcs(surface_layer=1)
    
button6.on_click(on_button_clicked6)

## Saving a larval shape <a id='section_saveshape'></a>

In [7]:
# Create and display a FileChooser widget
fc_s = FileChooser()
fc_s.filter_pattern = '*.pickle'
fc_s.title = '<b>Choose a filename for this larval morphology:</b>'
fc_s.default_filename = 'new_morph.pickle'

display(fc_s)

FileChooser(path='/home/dg/EBP/quant-org-bio/subrepos/biomechanics/ChimeraSwim', filename='new_morph.pickle', …

In [8]:
button3 = widgets.Button(description="Save file")
output3 = widgets.Output()

buttonsS = widgets.HBox([button3, output3])
display(buttonsS)

@output3.capture()
def on_button_clicked3(b):
    with open(fc_s.selected, 'wb') as handle:
        #print(f'Saving morphology as {fc_s.selected}')
        Mnew.clear_big_arrays()
        pickle.dump(Mnew, handle, protocol=pickle.HIGHEST_PROTOCOL)
        print(f'Saved morphology as {fc_s.selected}')

button3.on_click(on_button_clicked3)